## **EDA - Situación Problema: Wall Street Pulse**

Juan Pablo Espinosa Maciel | A01646333

---

**Objetivo:** Implementar un clasificador naïve Bayes para categorizar el nivel de impacto de noticias financieras a partir de sus titulares. A través de un conjunto de datos que combina texto no estructurado con indicadores como sentimiento, sector e índices bursátiles, construirán un pipeline de procesamiento de lenguaje natural (NLP) en R para predecir cómo estas variables repercuten en el mercado financiero.

In [ ]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print('Se importaron correctamente las librerías.')

In [ ]:
data = pd.read_csv('../data/raw/financial_news.csv')

data.head()

### 1. ¿Qué datos tenemos?

In [ ]:
# Dimensiones y tipos de datos
print(data.shape)

data.info()

In [ ]:
# Resumen de variables numéricas y valores únicos por columna
display(data.describe().round(2))

data.nunique()

In [ ]:
# Distribución de la variable objetivo y del sentimiento
orden = ['Low', 'Medium', 'High']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=data, x='Impact_Level', order=orden, ax=axes[0])
sns.countplot(data=data, x='Sentiment', ax=axes[1])

plt.tight_layout()
plt.show()

### 2. ¿Qué información no sirve?

In [ ]:
# Titulares únicos y repeticiones
print('Titulares distintos:', data['Headline'].nunique())

print('Palabras por titular:')
print(data['Headline'].str.split().str.len().describe().round(2))

data['Headline'].value_counts().head(10)

In [ ]:
# Un mismo titular aparece con distintos niveles de impacto
tabla = pd.crosstab(data['Headline'], data['Impact_Level'])[orden]

print('Aciertos máximos usando solo el titular:', round(tabla.max(axis=1).sum() / tabla.values.sum(), 3))
tabla.head(10)

In [ ]:
# Proporción de cada nivel de impacto según el sentimiento
pd.crosstab(data['Sentiment'], data['Impact_Level'], normalize='index').round(2)[orden]

In [ ]:
# Indicadores numéricos por nivel de impacto
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=data, x='Impact_Level', y='Index_Change_Percent', order=orden, ax=axes[0])
sns.boxplot(data=data, x='Impact_Level', y='Trading_Volume', order=orden, ax=axes[1])
plt.tight_layout()
plt.show()

**Columnas que no aportan al objetivo:**
- `News_Url`: enlace de referencia, no describe la noticia.
- `Date`: el objetivo no es temporal.
- `Source` y `Related_Company`: no dependen del titular ni del impacto.

`Headline` (entrada) e `Impact_Level` (objetivo) son las columnas centrales. `Sentiment`, `Sector`, `Market_Event` y los indicadores numéricos se conservan como complemento.

## 3. ¿Qué datos no tenemos y cómo los tratamos?

| Columna | Tratamiento |
|---|---|
| `Headline` | Eliminar la fila: es la entrada del clasificador. |
| `Sentiment` | Rellenar con `Unknown`. |
| `Index_Change_Percent` | Rellenar con la mediana. |
| `News_Url` | La columna se descarta. |

In [ ]:
# Nulos por columna
nulos = pd.DataFrame({'nulos': data.isna().sum(),
                      'porcentaje': (data.isna().mean() * 100).round(2)})
print('Filas duplicadas:', data.duplicated().sum())
nulos[nulos['nulos'] > 0]

In [ ]:
# Tratamiento de nulos
df = data.drop(columns=['News_Url', 'Date', 'Source', 'Related_Company'])

df = df.dropna(subset=['Headline'])                      # sin titular no hay texto que clasificar
df['Sentiment'] = df['Sentiment'].fillna('Unknown')      # se conserva como categoría propia
df['Index_Change_Percent'] = df['Index_Change_Percent'].fillna(df['Index_Change_Percent'].median())

print(df.shape)
df.isna().sum()

In [ ]:
df.to_csv('../data/interim/financial_news_processed.csv', index= False)

## 4. Descubrimientos

- **Variable objetivo balanceada:** `Impact_Level` se reparte casi en tercios (Medium 1,039, Low 1,020, High 965), por lo que no hace falta balancear clases.
- **Poca variedad de texto:** solo hay 50 titulares distintos en 3,024 filas, cada uno repetido ~60 veces. Esto limita el vocabulario del clasificador.
- **Etiquetas inconsistentes:** un mismo titular aparece con los tres niveles de impacto, sentimientos y sectores distintos. Si se asigna a cada titular su nivel más frecuente, se acierta solo ~41%, contra ~34% de adivinar la clase mayoritaria.
- **Sin relación aparente con otras variables:** la proporción de High/Medium/Low es prácticamente igual (~1/3) en cada sentimiento, y `Index_Change_Percent` y `Trading_Volume` tienen distribuciones casi idénticas en los tres niveles.
- **Datos faltantes bajos:** ~5% en cuatro columnas, sin filas duplicadas. Tras la limpieza quedan 2,876 registros sin nulos.
- **Implicación:** el conjunto parece sintético con etiquetas asignadas de forma aleatoria, así que se debe esperar un desempeño modesto del naïve Bayes.